In [ ]:

# ============================================================
# BANK CUSTOMER CHURN PREDICTION
# MILESTONE 2 & 3: DATA CLEANING AND PREPROCESSING
# ============================================================

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("=" * 60)
print("DATA CLEANING AND PREPROCESSING")
print("=" * 60)

# ------------------------------------------------------------
# 1. LOAD DATASET
# ------------------------------------------------------------

dataset_path = "Churn_Modelling_Uncleaned.csv"

if not os.path.exists(dataset_path):
    try:
        from google.colab import files
        print("Upload Churn_Modelling_Uncleaned.csv")
        uploaded = files.upload()

        if dataset_path not in uploaded:
            raise FileNotFoundError(
                "Please upload Churn_Modelling_Uncleaned.csv"
            )
    except ImportError:
        raise FileNotFoundError(
            "Place the original CSV file beside this notebook."
        )

df = pd.read_csv(dataset_path)

print("\nOriginal dataset shape:", df.shape)
display(df.head())

# ------------------------------------------------------------
# 2. DATA QUALITY CHECK
# ------------------------------------------------------------

print("\nMissing values before cleaning:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

# ------------------------------------------------------------
# 3. CLEAN DATA
# ------------------------------------------------------------

df_clean = df.copy()

# Remove duplicate rows
df_clean = df_clean.drop_duplicates().copy()

# Remove identifier columns that do not help predict churn
columns_to_drop = [
    col for col in ["RowNumber", "CustomerId", "Surname"]
    if col in df_clean.columns
]

df_clean.drop(columns=columns_to_drop, inplace=True)

print("\nRemoved identifier columns:", columns_to_drop)
print("Shape after cleaning:", df_clean.shape)

# Save the cleaned dataset.
# Missing values, if any, will be imputed later using
# training data only to avoid data leakage.
df_clean.to_csv("Churn_Modelling_Cleaned.csv", index=False)

print("Saved: Churn_Modelling_Cleaned.csv")

# ------------------------------------------------------------
# 4. FEATURE ENGINEERING
# ------------------------------------------------------------

df_featured = df_clean.copy()

if "Age" in df_featured.columns:
    df_featured["AgeGroup"] = pd.cut(
        df_featured["Age"],
        bins=[0, 30, 45, 60, np.inf],
        labels=["Young", "Adult", "Middle_Aged", "Senior"],
        right=True
    )

if "Balance" in df_featured.columns:
    df_featured["HasBalance"] = (
        df_featured["Balance"] > 0
    ).astype(int)

print("\nFeature engineering completed.")
print("New features: AgeGroup and HasBalance (where applicable)")
display(df_featured.head())

df_featured.to_csv("Churn_Modelling_Featured.csv", index=False)
print("Saved: Churn_Modelling_Featured.csv")

# ------------------------------------------------------------
# 5. SEPARATE FEATURES AND TARGET
# ------------------------------------------------------------

target = "Exited"

if target not in df_featured.columns:
    raise ValueError("Target column 'Exited' was not found.")

X = df_featured.drop(columns=[target])
y = df_featured[target]

# ------------------------------------------------------------
# 6. TRAIN-TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))

print(
    "Train/test row overlap:",
    len(set(X_train.index).intersection(set(X_test.index)))
)

# ------------------------------------------------------------
# 7. IDENTIFY NUMERICAL AND CATEGORICAL FEATURES
# ------------------------------------------------------------

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nNumerical features:", numeric_features)
print("\nCategorical features:", categorical_features)

# ------------------------------------------------------------
# 8. PREPROCESSING PIPELINES
# ------------------------------------------------------------

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

# Fit ONLY on training data to prevent data leakage
X_train_processed = preprocessor.fit_transform(X_train)

# Use the fitted transformations on test data
X_test_processed = preprocessor.transform(X_test)

# ------------------------------------------------------------
# 9. CREATE MODEL-READY DATASETS
# ------------------------------------------------------------

feature_names = preprocessor.get_feature_names_out()

X_train_ready = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_ready = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

X_train_ready[target] = y_train
X_test_ready[target] = y_test

X_train_ready.to_csv("train_model_ready.csv", index=False)
X_test_ready.to_csv("test_model_ready.csv", index=False)

# ------------------------------------------------------------
# 10. FINAL CHECKS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PREPROCESSING COMPLETED")
print("=" * 60)

print("\nTraining dataset shape:", X_train_ready.shape)
print("Testing dataset shape:", X_test_ready.shape)

print("\nMissing values in training data:",
      X_train_ready.isnull().sum().sum())

print("Missing values in testing data:",
      X_test_ready.isnull().sum().sum())

print("\nTarget distribution in training data:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTarget distribution in testing data:")
print(y_test.value_counts(normalize=True).round(3))

print("\nFiles saved:")
print("- Churn_Modelling_Cleaned.csv")
print("- Churn_Modelling_Featured.csv")
print("- train_model_ready.csv")
print("- test_model_ready.csv")

print("\nNo machine learning model has been trained yet.")
print("The data is ready for the next milestone.")


DATA CLEANING AND PREPROCESSING

Original dataset shape: (10018, 14)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619.0,France,Female,42.0,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608.0,Spain,Female,41.0,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502.0,France,Female,42.0,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699.0,France,Female,39.0,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850.0,Spain,Female,43.0,2,125510.82,1,1,1,79084.10,0



Missing values before cleaning:
RowNumber            0
CustomerId           0
Surname              0
CreditScore        200
Geography            0
Gender               0
Age                250
Tenure               0
Balance              0
NumOfProducts        0
HasCrCard            0
IsActiveMember       0
EstimatedSalary    100
Exited               0
dtype: int64

Duplicate rows: 18

Removed identifier columns: ['RowNumber', 'CustomerId', 'Surname']
Shape after cleaning: (10000, 11)
Saved: Churn_Modelling_Cleaned.csv

Feature engineering completed.
New features: AgeGroup and HasBalance (where applicable)


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,AgeGroup,HasBalance
0,619.0,France,Female,42.0,2,0.00,1,1,1,101348.88,1,Adult,0
1,608.0,Spain,Female,41.0,1,83807.86,1,0,1,112542.58,0,Adult,1
2,502.0,France,Female,42.0,8,159660.80,3,1,0,113931.57,1,Adult,1
3,699.0,France,Female,39.0,1,0.00,2,0,0,93826.63,0,Adult,0
4,850.0,Spain,Female,43.0,2,125510.82,1,1,1,79084.10,0,Adult,1


Saved: Churn_Modelling_Featured.csv

Training rows: 8000
Testing rows: 2000
Train/test row overlap: 0

Numerical features: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'HasBalance']

Categorical features: ['Geography', 'Gender', 'AgeGroup']

PREPROCESSING COMPLETED

Training dataset shape: (8000, 19)
Testing dataset shape: (2000, 19)

Missing values in training data: 0
Missing values in testing data: 0

Target distribution in training data:
Exited
0    0.796
1    0.204
Name: proportion, dtype: float64

Target distribution in testing data:
Exited
0    0.796
1    0.204
Name: proportion, dtype: float64

Files saved:
- Churn_Modelling_Cleaned.csv
- Churn_Modelling_Featured.csv
- train_model_ready.csv
- test_model_ready.csv

No machine learning model has been trained yet.
The data is ready for the next milestone.
